# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 dataset using the `mlcroissant` library, referencing dataset components by their `@id` fields for consistency and reproducibility.

### Dataset Source
The dataset source is provided as a Croissant schema URL.

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata_obj = dataset.metadata

# Print basic information from the metadata
print("Dataset Title: ", metadata_obj.name)
print("Description: ", metadata_obj.description)
print("Published Date: ", getattr(metadata_obj, 'datePublished', 'N/A'))
print("Authors: ", getattr(metadata_obj, 'author', 'N/A'))
print("Version: ", getattr(metadata_obj, 'version', 'N/A'))

## 2. Data Overview
Review available record sets, fields, and their `@id` identifiers. Each entity in Croissant has a unique `@id` that will be used for referencing in downstream operations.

In [ ]:
# List all available Record Sets and their field @id's

record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record set(s):\n")
all_record_set_ids = []
for rs in record_sets:
    print(f"- Record Set name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {getattr(rs, 'description', '')}")
    field_ids = []
    if hasattr(rs, 'fields'):
        print(f"  Fields:")
        for field in rs.fields:
            print(f"    - {field.name} (@id: {field.id})")
            field_ids.append(field.id)
    print("")
    all_record_set_ids.append(rs.id)

# If there's only one record set, show preview of a few records
if len(record_sets) > 0:
    chosen_rs_id = record_sets[0].id
    print(f"First 2 records in Record Set '@id': {chosen_rs_id}")
    for i, rec in enumerate(dataset.records(record_set=chosen_rs_id)):
        print(rec)
        if i >= 1:
            break

## 3. Data Extraction
Load data from one or more record sets into DataFrames for analysis. Use the record set and field `@id`s obtained in the overview above.

In [ ]:
# Prepare dictionary to hold DataFrames for each record set
import collections

dataframes = collections.OrderedDict()

for rs in dataset.record_sets:
    print(f"Loading records for Record Set: {rs.name} (@id: {rs.id})")
    records = list(dataset.records(record_set=rs.id))
    # There may be nested dicts for fields, let's normalize
    df = pd.json_normalize(records)
    dataframes[rs.id] = df
    print(f"Shape: {df.shape}, Columns: {df.columns.tolist()}")
    print(df.head(2), '\n')

# Choose the primary record set (the largest one if unsure)
if len(dataframes):
    main_rs_id = max(dataframes, key=lambda x: dataframes[x].shape[0])
    print(f"Selected primary record set for analysis: {main_rs_id}")
    print(f"Columns: {dataframes[main_rs_id].columns.tolist()}")
    dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's filter, normalize, and group the data with reference to the proper field (column) `@id`s, preparing for downstream analysis.

Steps include filtering rows, normalizing a numeric field, and grouping.

In [ ]:
# Determine which numeric field to analyze from the columns
df = dataframes[main_rs_id]
print("Available columns:", df.columns.tolist())

# Try to automatically pick a numeric field, fallback to user suggestion
import numpy as np
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if numeric_cols:
    numeric_field = numeric_cols[0]
    print(f"Using numeric field: {numeric_field}")
else:
    # Try to find a field containing 'abundance' or 'peptide' in its name
    candidates = [c for c in df.columns if any(x in c.lower() for x in ["abundance", "peptide", "coverage", "mw"])]
    numeric_field = candidates[0] if candidates else df.columns[0]
    print(f"Defaulting to first field: {numeric_field}")

# Remove missing values for chosen field
df_numeric = pd.to_numeric(df[numeric_field], errors='coerce')
mask = ~df_numeric.isna()
df = df[mask]

threshold = df[numeric_field].mean()
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean): {len(filtered_df)} records")
print(filtered_df.head(3))

# Normalize the field (Z-score)
filtered_df = filtered_df.copy()
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"].copy()].head())

# Try grouping by another field if categorical columns exist
categoricals = [c for c in df.columns if df[c].nunique() < 10 and c != numeric_field]
group_field = categoricals[0] if categoricals else None
if group_field:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data by {group_field}:")
    print(grouped_df[[numeric_field, f"{numeric_field}_normalized"]].head())

## 5. Visualization
Let's visualize the distribution of the selected numeric field and (if available) its normalized form.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 4))
sns.histplot(filtered_df[numeric_field], bins=30, kde=True, color='skyblue')
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

if f"{numeric_field}_normalized" in filtered_df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[f"{numeric_field}_normalized"], bins=30, kde=True, color='salmon')
    plt.title(f"Normalized Distribution of {numeric_field}")
    plt.xlabel(f"{numeric_field}_normalized")
    plt.ylabel("Count")
    plt.show()

# If a group field exists, compare group means
if group_field:
    plt.figure(figsize=(7, 4))
    sns.barplot(x=group_field, y=numeric_field, data=filtered_df, ci=None)
    plt.title(f"{numeric_field} by {group_field}")
    plt.ylabel(numeric_field)
    plt.xlabel(group_field)
    plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to load and process the FAIR^2 dataset using `mlcroissant`, referencing all data structures by their Croissant `@id`s. You can now extend this notebook to perform specific analyses on protein abundance, peptide counts, molecular weights, or other fields relevant to your research question.